# 03 Advanced EDA

This notebook goes beyond plotting. Each section is framed as a business question and uses the data to produce an answer that can guide pricing, listing strategy, and market targeting.

## 1. Setup
Import the libraries needed for analysis and visualization.

In [ ]:
import pandas as pd
import plotly.express as px

pd.set_option('display.max_columns', None)

## 2. Load Data
We use the processed Airbnb data so the analysis focuses on business interpretation instead of cleaning noise.

In [ ]:
df = pd.read_csv('../data/processed/merged_data.csv')
df.head()

## 3. Data Snapshot
Before answering business questions, we confirm the shape of the market and the available fields.

In [ ]:
display(df.info())
display(df.describe(include='all').T.head(20))

## 4. Business Question 1: Where should hosts focus to maximize revenue?
We look for neighborhoods with strong pricing and availability balance, because that combination usually drives revenue potential.

In [ ]:
if 'price' not in df.columns and 'price_x' in df.columns:
    df['price'] = df['price_x']
elif 'avg_daily_price' in df.columns and 'price' not in df.columns:
    df['price'] = df['avg_daily_price']

if 'availability_365' not in df.columns and 'available_days' in df.columns:
    df['availability_365'] = df['available_days']

neighborhood_col = 'neighbourhood_cleansed' if 'neighbourhood_cleansed' in df.columns else None
if neighborhood_col is None:
    raise KeyError('Expected a neighborhood column such as neighbourhood_cleansed.')

question_1 = (
    df.groupby(neighborhood_col, as_index=False)
      .agg(
          listing_count=('id', 'count') if 'id' in df.columns else ('price', 'size'),
          avg_price=('price', 'mean'),
          median_price=('price', 'median'),
          avg_availability=('availability_365', 'mean')
      )
      .sort_values(['avg_price', 'listing_count'], ascending=[False, False])
)
question_1.head(10)

In [ ]:
fig = px.bar(
    question_1.head(15),
    x='avg_price',
    y=neighborhood_col,
    orientation='h',
    title='Top Neighborhoods by Average Price',
    labels={'avg_price': 'Average Price', neighborhood_col: 'Neighborhood'}
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()

**Business takeaway:** Prioritize the neighborhoods at the top of the table for premium positioning, but validate that the listing count is large enough to support demand.

## 5. Business Question 2: Which room types have the strongest pricing power?
Room type mix tells us whether a host should compete on premium private spaces or volume-driven shared inventory.

In [ ]:
room_col = 'room_type' if 'room_type' in df.columns else None
if room_col is None:
    raise KeyError('Expected a room_type column.')

room_summary = (
    df.groupby(room_col, as_index=False)
      .agg(
          listing_count=('price', 'size'),
          avg_price=('price', 'mean'),
          avg_availability=('availability_365', 'mean')
      )
      .sort_values('avg_price', ascending=False)
)
room_summary

In [ ]:
fig = px.scatter(
    room_summary,
    x='avg_availability',
    y='avg_price',
    size='listing_count',
    color=room_col,
    title='Room Type Pricing vs Availability',
    labels={'avg_availability': 'Average Availability', 'avg_price': 'Average Price'}
)
fig.show()

**Business takeaway:** If a room type combines higher price with acceptable availability, it is a stronger candidate for revenue optimization.

## 6. Business Question 3: What price bands dominate the market?
Understanding the distribution of prices helps identify entry-level, mid-market, and premium segments.

In [ ]:
price_series = df['price'].dropna()
df['price_band'] = pd.qcut(price_series.rank(method='first'), q=3, labels=['Low', 'Mid', 'High'])
price_band_summary = df['price_band'].value_counts().sort_index().reset_index()
price_band_summary.columns = ['price_band', 'listing_count']
price_band_summary

In [ ]:
fig = px.pie(
    price_band_summary,
    names='price_band',
    values='listing_count',
    title='Market Share by Price Band'
)
fig.show()

**Business takeaway:** The largest price band shows where most hosts compete, which is useful for positioning and differentiation.

## 7. Business Question 4: Does review activity signal market demand?
Listings with more reviews may indicate stronger guest interest or better visibility.

In [ ]:
review_col = 'number_of_reviews' if 'number_of_reviews' in df.columns else None
if review_col is None and 'review_count' in df.columns:
    review_col = 'review_count'
if review_col is None:
    raise KeyError('Expected number_of_reviews or review_count column.')

corr_df = df[[review_col, 'price']].dropna()
correlation = corr_df[review_col].corr(corr_df['price'])
print(f'Correlation between {review_col} and price: {correlation:.3f}')

fig = px.scatter(
    corr_df.sample(min(len(corr_df), 2000), random_state=42),
    x=review_col,
    y='price',
    trendline='ols',
    title='Review Activity vs Price'
)
fig.show()

**Business takeaway:** If the relationship is weak, review volume should be treated more as a visibility metric than a pricing driver.

## 8. Conclusion
This notebook answers practical Airbnb business questions about market positioning, room-type economics, price segmentation, and demand signals. The next step is to turn these insights into features for modeling and host recommendations.